In [43]:
## loadding required libs
import pandas as pd
import geopandas as gpd
import numpy as np
from IPython.display import HTML, display
import time
import ipywidgets as widgets
import ipywidgets as w
from IPython.display import Javascript

## set working directory
import os
os.chdir("/Users/yuxin/Dropbox/DDDI/Illegal-Dumping")

In [6]:
## load illegal dumping data
## **please change the working directory to yours**

## Got updated data from 311 team on 8/20/2025

file_paths = {
    "ppr311_21":"/Users/yuxin/Library/CloudStorage/Box-Box/Illegal-Dumping/data/311/PPR3112020_to_2022/PPR311DumpingDataforDDDI_CY2021.csv",
    "ppr311_22":"/Users/yuxin/Library/CloudStorage/Box-Box/Illegal-Dumping/data/311/PPR3112020_to_2022/PPR311DumpingDataforDDDI_CY2022.csv",
    "ppr311_23":"/Users/yuxin/Library/CloudStorage/Box-Box/Illegal-Dumping/data/311/PPR3112023_to_2025/PPR311DumpingDataforDDDI_CY2023.csv",
    "ppr311_24":"/Users/yuxin/Library/CloudStorage/Box-Box/Illegal-Dumping/data/311/PPR3112023_to_2025/PPR311DumpingDataforDDDI_CY2024.csv",
    "ppr311_25":"/Users/yuxin/Library/CloudStorage/Box-Box/Illegal-Dumping/data/311/PPR3112023_to_2025/PPR311DumpingDataforDDDI_CY2025.csv",
}

rename_map = {
        "Date/Time Opened" :"start_time",
        "Date/Time Closed" : "close_time",
        "Service Request Number" : "request_id",
        "Service Request Type" : "type",
        "Department" : "department",
        "Address/Intersection" : "address",
        "ZipCode" : "zipcode",
        "Status" : "status",
        "Problem Category" : "problem_category",
        "Problem" : "problem",
        "Location Type" : "location_type",
        "Park District" : "park_district",
        "Large Crew&Heavy Machinery To Clean Up?" : "large_crew",
        "Type of Materials" : "material_type",
        "Condition of Materials" : "material_condition",
        "Description" : "description",
        "Centerline (Latitude)" : "lat",
        "Centerline (Longitude)" : "lon",
        "Case ID" : "case_id",
        "Parent Case ID" : "parent_case_id",
        "Mobile App Photos" : "media_url"
}

for var_name, path in file_paths.items():
    with open(path, encoding="utf-8", errors="replace") as f:
        df = pd.read_csv(f).rename(columns=rename_map)
        
        # ==== add year column ====
        year = int(var_name.split("_")[1]) + 2000  # e.g., "ppr311_21" → 21 → 2021
        df["year"] = year

        # ==== convert time columns to yyyy/mm/dd HH:MM ====
        for col in ["start_time", "close_time"]:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors="coerce")
                df[col] = df[col].dt.strftime("%Y/%m/%d %H:%M")

        # ==== clean coords ====
        df[["lon", "lat"]] = df[["lon", "lat"]].apply(pd.to_numeric, errors="coerce")
        df.loc[(df["lon"] == 0) & (df["lat"] == 0), ["lon", "lat"]] = np.nan
        deg_mask = df["lon"].between(-76, -74) & df["lat"].between(39, 41)
        df = df[deg_mask].dropna(subset=["lon", "lat"])

        # ==== build geometry ====
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df["lon"], df["lat"]),
            crs=4326
        ).to_crs(26918)

        globals()[var_name] = gdf


/var/folders/nr/q7sysy9j07x_5smd2gfx3j380000gn/T/ipykernel_9513/2111686087.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")
/var/folders/nr/q7sysy9j07x_5smd2gfx3j380000gn/T/ipykernel_9513/2111686087.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")
/var/folders/nr/q7sysy9j07x_5smd2gfx3j380000gn/T/ipykernel_9513/2111686087.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")
/var/folders/nr/q7sysy9j07x_5smd2gfx3j38000

In [7]:
## concact 2021, 2022, 2023, and 2024 data
illegal_dumping = pd.concat([
    ppr311_21,
    ppr311_22,
    ppr311_23,
    ppr311_24,
    ppr311_25
])

In [13]:
## filter out pending cases

## loadding pending cased list
pending_id = pd.read_csv('output/picked_pending.csv', header=None, names=['request_id'])

## merge with illegal dumping data
pending_cases = illegal_dumping[illegal_dumping['request_id'].isin(pending_id['request_id'])]

In [40]:
dfv = pending_cases[["request_id","type","status","description"]]
dfv["confirm"] = ""


/var/folders/nr/q7sysy9j07x_5smd2gfx3j380000gn/T/ipykernel_9513/3144669998.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfv["confirm"] = ""


In [45]:
## export the data to csv
dfv.to_csv("output/dfv.csv", index=False)